In [6]:
!pip install pypdf

     ---------------------------------------- 0.0/333.7 kB ? eta -:--:--
     -------------------------------------- 333.7/333.7 kB 6.9 MB/s eta 0:00:00



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
!pip install faiss-cpu

     ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
      --------------------------------------- 0.3/18.9 MB 8.9 MB/s eta 0:00:03
     - -------------------------------------- 0.9/18.9 MB 11.3 MB/s eta 0:00:02
     --- ------------------------------------ 1.8/18.9 MB 14.0 MB/s eta 0:00:02
     ----- ---------------------------------- 2.8/18.9 MB 16.2 MB/s eta 0:00:01
     -------- ------------------------------- 4.2/18.9 MB 19.3 MB/s eta 0:00:01
     ------------ --------------------------- 5.7/18.9 MB 20.3 MB/s eta 0:00:01
     --------------- ------------------------ 7.1/18.9 MB 21.6 MB/s eta 0:00:01
     ------------------ --------------------- 8.5/18.9 MB 23.7 MB/s eta 0:00:01
     -------------------- ------------------ 10.0/18.9 MB 23.6 MB/s eta 0:00:01
     ------------------------ -------------- 11.6/18.9 MB 29.7 MB/s eta 0:00:01
     --------------------------- ----------- 13.2/18.9 MB 32.7 MB/s eta 0:00:01
     ------------------------------ -------- 14.8


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# from pypdf import PdfReader

# Put below code in  rag_core.py

In [12]:
# rag_core.py

from langchain_community.document_loaders import PyPDFLoader # install pypdf
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS # install faiss-gpu

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
import tempfile

# -------- LOAD + SPLIT --------
def load_and_split_pdfs(files):
    all_docs = []

    for file in files:
        # Handle both Streamlit file and local file
        if hasattr(file, "read"):
            # Streamlit file
            with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
                tmp.write(file.read())
                file_path = tmp.name
        else:
            # Local file path
            file_path = file

        loader = PyPDFLoader(file_path)
        docs = loader.load()

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=100
        )

        split_docs = splitter.split_documents(docs)
        all_docs.extend(split_docs)

    return all_docs

# -------- BUILD VECTOR STORE --------
def build_vector_store(docs, api_key):
    embeddings = OpenAIEmbeddings(openai_api_key=api_key)

    vectorstore = FAISS.from_documents(docs, embeddings)

    return vectorstore


# -------- RETRIEVER --------
def get_retriever(vectorstore):
    return vectorstore.as_retriever(search_kwargs={"k": 3})


# -------- ASK LLM --------
def ask_llm(question, retriever, api_key):
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0,
        openai_api_key=api_key
    )

    docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
    Answer using the context below. Keep it short.

    Context:
    {context}

    Question: {question}
    """

    response = llm.invoke(prompt)

    return response.content, docs


In [13]:
if __name__ == "__main__":

    import os

    #  Set your API key here (temporary for testing)
    api_key = "key_here"

    #  Provide path to a local PDF
    test_pdf = "document-loxford-company.pdf"

    if not os.path.exists(test_pdf):
        print(" PDF file not found. ")
        exit()

    print(" Loading and splitting PDF...")
    docs = load_and_split_pdfs([open(test_pdf, "rb")])

    print(f" Total chunks created: {len(docs)}")

    print(" Building vector store...")
    vectorstore = build_vector_store(docs, api_key)

    retriever = get_retriever(vectorstore)

    # Test question
    question = "What is this document about?"

    print(f"\n Question: {question}")

    answer, retrieved_docs = ask_llm(question, retriever, api_key)

    print("\n Retrieved Chunks:")
    for i, d in enumerate(retrieved_docs):
        print(f"\n--- Chunk {i+1} ---")
        print(d.page_content[:200])

    print("\n Final Answer:")
    print(answer)


📄 Loading and splitting PDF...
✅ Total chunks created: 4
🔍 Building vector store...

❓ Question: What is this document about?

📚 Retrieved Chunks:

--- Chunk 1 ---
• Cloud & DevOps Services 
Scalable infrastructure deployment using AWS, Azure, and GCP . 
• Process Automation 
Workflow automation using AI agents and robotic process automation 
(RPA). 
Some of its

--- Chunk 2 ---
Recent Developments  
In 2026, the company launched its first Agentic AI platform, enabling autonomous 
decision-making systems for enterprise clients. It has also partnered with several 
Fortune 500 

--- Chunk 3 ---
Loxford Technologies – Company Overview 
Loxford Technologies is a global technology solutions provider specializing in 
artificial intelligence, data analytics, and enterprise automation. Founded in 

🤖 Final Answer:
This document provides an overview of Loxford Technologies, a global technology solutions provider specializing in artificial intelligence, data analytics, and enterprise automation.